# Tutorial: LoRA, Parameter-Efficient Fine-tuning
**Model:** `distilbert-base-uncased`
**Task:** IMDb Sentiment Classification

---

## Learning Objectives

You will be able to:
1. **Explain the motivation** for parameter-efficient fine-tuning and why full fine-tuning is impractical at scale.
2. **Understand LoRA's mathematical intuition**: the low-rank decomposition `ΔW = BA`.
3. **Apply LoRA to DistilBERT** using the Hugging Face PEFT library with just a few lines of configuration.
4. **Compare three strategies** — linear probing, LoRA, and full fine-tuning — in terms of trainable parameters and accuracy.

---

Run on **Google Colab** with a free T4 GPU: `Runtime → Change runtime type → T4 GPU`.

## 1. Setup

In [ ]:
# peft = Parameter-Efficient Fine-Tuning library by Hugging Face
!pip install -q transformers datasets peft torch accelerate

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > GPU to enable it.")

## 2. Motivation: Why Not Just Fine-tune Everything?

In the probing tutorial, we saw that probing with 1,538 parameters gives ~80% accuracy on sentiment. Full fine-tuning would update all ~66 million DistilBERT parameters and likely reach ~94%. But how about the cost?

**Memory cost of full fine-tuning:**

The Adam optimizer maintains **two extra tensors per parameter** (first and second moment estimates). So training DistilBERT requires roughly:

```
model weights:   66M × 4 bytes  = 264 MB
gradients:       66M × 4 bytes  = 264 MB
Adam state (×2): 66M × 8 bytes  = 528 MB
──────────────────────────────────────────
Total:                          ≈ 1 GB  (just for DistilBERT)
```

Scale this to GPT-3 (175 billion parameters) and full fine-tuning requires **~2.1 TB of memory** — impossible on any single machine.

**Parameter-efficient fine-tuning (PEFT)** methods freeze the base model and add a small number of trainable parameters. The goal: match full fine-tuning accuracy while training a tiny fraction of the parameters.

| Method | Trainable Params | Notes |
|---|---|---|
| Full fine-tuning | ~66M (100%) | All backbone weights updated |
| LoRA (r=8) | ~300K (<1%) | Low-rank adapter matrices |
| Linear probe | ~1.5K (0.002%) | Only classification head |

## 3. LoRA: Mathematical Intuition

### The Core Idea

During pretraining, a weight matrix **W** (e.g., the query projection in attention, shape `768 × 768`) was learned. During fine-tuning, we want to update W by some change **ΔW**.

LoRA's key empirical observation: **the update ΔW has low intrinsic rank**. This means ΔW can be well-approximated by the product of two small matrices:

$$\Delta W = B \cdot A$$

where:
- **A** has shape `(r, d)` — projects down to rank `r`
- **B** has shape `(d, r)` — projects back up to `d`
- `r << d` (e.g., r=8, d=768)

```
                  d=768
         ┌──────────────────┐
         │                  │   W (frozen)   shape: 768 × 768
    d    │        W         │
  =768   │   (no gradient)  │
         │                  │
         └──────────────────┘
                 +
         ┌──────────────────┐
    r    │  B  (768 × r)    │   B (trainable)   shape: 768 × r
   =8    └──────┬───────────┘
                │
         ┌──────┴───────────┐
    d    │  A  (r × 768)    │   A (trainable)   shape: r × 768
  =768   └──────────────────┘
```

**Parameter count comparison for one projection matrix:**
- Full W: `768 × 768 = 589,824` parameters
- A + B: `(8 × 768) + (768 × 8) = 6,144 + 6,144 = 12,288` parameters — **48× fewer!**

### The Modified Forward Pass

The layer output becomes:

$$h = W x + \frac{\alpha}{r} \cdot B A x$$

- `W x` is the original frozen computation (no gradient storage needed).
- `(α/r) · BAx` is the trainable adapter contribution.
- **α (lora_alpha)** is a scaling hyperparameter; the convention `α = 2r` gives a scale of 2.0.

### Initialization

At initialization, **B is set to all zeros**. This means the adapter contribution `BAx = 0` at the start of training — the model begins fine-tuning from exactly the pretrained weights, with no disruption.

## 4. Load and Subsample the IMDb Dataset

In [ ]:
# Load IMDb — same subsample as Tutorial 1 (seed=42 for reproducibility)
raw = load_dataset("imdb")
train_data = raw["train"].shuffle(seed=42).select(range(8000))
test_data  = raw["test"].shuffle(seed=42).select(range(500))

print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_tok = train_data.map(tokenize, batched=True, remove_columns=["text"])
test_tok  = test_data.map(tokenize,  batched=True, remove_columns=["text"])

train_tok.set_format("torch")
test_tok.set_format("torch")

train_loader = DataLoader(train_tok, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_tok,  batch_size=64)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 5. Apply LoRA with the PEFT Library

Hugging Face's `peft` library makes applying LoRA a matter of configuration. Three key components:

1. **`LoraConfig`** — specifies rank `r`, scaling `lora_alpha`, dropout, and which modules to target.
2. **`get_peft_model(base_model, config)`** — wraps the base model, replacing targeted modules with LoRA-enhanced versions.
3. **`model.print_trainable_parameters()`** — immediately shows how many params are trainable.

**Important distinction — `target_modules` vs `modules_to_save`:**

| Parameter | What it does |
|---|---|
| `target_modules` | Layers that get LoRA adapters (A and B matrices added alongside frozen W) |
| `modules_to_save` | Layers that are fully trained and saved in the adapter checkpoint (not decomposed) |

The classification head (`classifier`) should be in `modules_to_save`, not `target_modules` — we want to fully train it, not decompose it with LoRA.

**DistilBERT attention module names:**  
DistilBERT's attention projections are named `q_lin`, `k_lin`, `v_lin`, `out_lin`. Following the original LoRA paper, we target `q_lin` and `v_lin` (query and value projections).

In [ ]:
# Load base model with a classification head (num_labels=2 for binary sentiment)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

total_base = sum(p.numel() for p in base_model.parameters())
print(f"Base model total parameters: {total_base:,}")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,          # sequence classification task
    r=8,                                  # rank of the decomposition
    lora_alpha=16,                        # scaling factor; effective scale = alpha/r = 2.0
    lora_dropout=0.1,                     # dropout on adapter activations
    target_modules=["q_lin", "v_lin"],   # DistilBERT attention projections
    modules_to_save=["classifier"],       # fully train (not LoRA-decompose) the head
    bias="none"                           # don't add trainable biases
)

lora_model = get_peft_model(base_model, lora_config)

# This single line shows the power of PEFT:
lora_model.print_trainable_parameters()

In [ ]:
# (Optional diagnostic) — see which modules now have LoRA adapters
print("Modules with LoRA adapters:")
for name, module in lora_model.named_modules():
    if "lora_A" in name or "lora_B" in name:
        print(f"  {name}")

## 6. Training

The training loop looks nearly identical to standard fine-tuning. PEFT handles all the gradient masking internally — only LoRA parameters (A, B matrices) and `modules_to_save` parameters (the classifier head) have `requires_grad=True`.

We use the Hugging Face pattern of passing `labels` directly to the model and using `outputs.loss` — this avoids reimplementing the cross-entropy loss.

**Note:** LoRA typically uses a higher learning rate than full fine-tuning (1e-4 to 3e-4 is common) because we're training a small adapter from scratch, not nudging a pretrained weight.

In [ ]:
# Optimizer only over trainable parameters (PEFT has already set requires_grad correctly)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()),
    lr=2e-4
)

EPOCHS = 3

for epoch in range(EPOCHS):
    lora_model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        # Pass labels to the model → outputs.loss is cross-entropy
        outputs = lora_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss   = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()   # gradients only flow into LoRA A/B + classifier
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(dim=-1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    acc      = correct / total
    print(f"  → Loss: {avg_loss:.4f} | Train Acc: {acc:.4f}")

## 7. Evaluate LoRA Model

In [ ]:
lora_model.eval()
correct, total = 0, 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask)
        correct += (outputs.logits.argmax(dim=-1) == labels).sum().item()
        total   += labels.size(0)

lora_acc = correct / total
print(f"\nLoRA Test Accuracy: {lora_acc:.4f} ({lora_acc*100:.1f}%)")
print("Expected range: ~91–93% on this 500-example test set")

## 8. Saving and Loading LoRA Adapters

One of LoRA's practical advantages: the **saved adapter is tiny** — it only contains the A and B matrices plus the `modules_to_save` weights. The base DistilBERT model (hundreds of MB) can be re-downloaded or shared separately and is never duplicated.

This enables:
- **Adapter swapping**: load the same base model and apply different adapters for different tasks at runtime.
- **Cheap storage**: store dozens of task-specific adapters without duplicating the base model.

In [ ]:
import os

ADAPTER_DIR = "./distilbert-imdb-lora"
lora_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")
print("\nFiles saved:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_kb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1024
    print(f"  {f:<40} {size_kb:.1f} KB")

# Note: adapter_model.safetensors contains only A, B matrices + classifier
# Compare this to the full DistilBERT model (~250 MB)

In [ ]:
# Reload: start from fresh base model, attach saved adapter
fresh_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)

loaded_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)
loaded_model.eval()

# Sanity check: run on a few examples
print("Quick sanity check on 3 test examples:\n")
sample = test_data.select(range(3))
for item in sample:
    enc = tokenizer(
        item["text"], return_tensors="pt",
        truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        logit = loaded_model(**enc).logits
    pred  = logit.argmax(dim=-1).item()
    label = item["label"]
    pred_str  = "POSITIVE" if pred  == 1 else "NEGATIVE"
    label_str = "POSITIVE" if label == 1 else "NEGATIVE"
    match = "✓" if pred == label else "✗"
    print(f"  {match} Predicted: {pred_str:<10} True: {label_str}")
    print(f"    Review: \"{item['text'][:80]}...\"\n")

## 9. Parameter Count Comparison: Three Strategies

Let's compute and visualize the exact parameter counts for all three fine-tuning strategies side-by-side.

In [ ]:
def count_params(model, only_trainable=False):
    """Count total or trainable parameters in a model."""
    if only_trainable:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


# ── Full fine-tuning ─────────────────────────────────────────────────────────
full_model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
full_total      = count_params(full_model)
full_trainable  = count_params(full_model, only_trainable=True)  # all params

# ── LoRA (current model) ─────────────────────────────────────────────────────
lora_total      = count_params(lora_model)
lora_trainable  = count_params(lora_model, only_trainable=True)

# ── Linear probe (conceptual — reproduced from Tutorial 1) ───────────────────
probe_backbone   = AutoModel.from_pretrained(MODEL_NAME)
probe_head       = nn.Linear(768, 2)
probe_total      = count_params(probe_backbone) + count_params(probe_head)
probe_trainable  = count_params(probe_head)  # 768*2 + 2 = 1,538

print("=" * 65)
print(f"{'Method':<22} {'Trainable':>12} {'Total':>12} {'% Trainable':>12}")
print("=" * 65)
print(f"{'Full Fine-tuning':<22} {full_trainable:>12,} {full_total:>12,} {100.0:>11.3f}%")
print(f"{'LoRA (r=8)':<22} {lora_trainable:>12,} {lora_total:>12,} {100*lora_trainable/lora_total:>11.3f}%")
print(f"{'Linear Probe':<22} {probe_trainable:>12,} {probe_total:>12,} {100*probe_trainable/probe_total:>11.3f}%")
print("=" * 65)

In [ ]:
methods    = ["Full\nFine-tuning", "LoRA\n(r=8)", "Linear\nProbe"]
trainable  = [full_trainable, lora_trainable, probe_trainable]
colors     = ["#e74c3c", "#3498db", "#2ecc71"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: Absolute count (log scale)
bars1 = ax1.bar(methods, trainable, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_yscale("log")
ax1.set_ylabel("Trainable Parameters (log scale)", fontsize=11)
ax1.set_title("Trainable Parameter Count", fontsize=13)
for bar, v in zip(bars1, trainable):
    ax1.text(bar.get_x() + bar.get_width()/2, v * 2,
             f"{v:,}", ha="center", va="bottom", fontsize=9)

# Right: Percentage of full model
pcts  = [100 * t / full_total for t in trainable]
bars2 = ax2.bar(methods, pcts, color=colors, edgecolor='white', linewidth=1.5)
ax2.set_ylabel("% of Full Model Parameters", fontsize=11)
ax2.set_title("Parameter Efficiency", fontsize=13)
for bar, v in zip(bars2, pcts):
    label = f"{v:.2f}%" if v >= 0.01 else f"{v:.4f}%"
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.05,
             label, ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 10. Accuracy + Efficiency Summary

Fill in your own results from both notebooks in the cell below.

In [ ]:
# ── Fill in your observed results from both notebooks ───────────────────────
# (Replace the approximate values below with your actual measurements)

results = [
    # (method_name,           test_acc,  trainable_params)
    ("Linear Probe",          0.83,      probe_trainable),
    ("LoRA (r=8)",            lora_acc,  lora_trainable),
    ("Full Fine-tuning (ref)",0.94,      full_trainable),   # reference only
]

print(f"{'Method':<26} {'Test Acc':>9} {'Trainable Params':>18}")
print("─" * 70)
for name, acc, params in results:
    print(f"{name:<26} {acc:>9.4f} {params:>18,}")

print("\nKey takeaway:")
print(f"  LoRA trains {lora_trainable/full_trainable*100:.2f}% of parameters")
print(f"  and achieves {lora_acc:.4f} accuracy vs. ~0.94 for full fine-tuning.")

**Key insight:** LoRA closes most of the gap between probing (83%) and full fine-tuning (94%) while training less than 1% of the parameters. This is the central value proposition of PEFT methods:

- Probing hits the **ceiling of what frozen representations linearly encode**.
- LoRA adapts the representations from within — allowing the model to reconfigure what information is encoded — without the full cost of fine-tuning all weights.
- The tradeoff curve (accuracy vs. trainable params) is fundamental to the field, and LoRA occupies the practical sweet spot for most tasks where the pretrained model already has relevant knowledge.

## 11. Exploration Exercises

Try these experiments to deepen your understanding:

---

### Exercise 1: Effect of rank `r`

Re-run Section 5 with `r=4` and `r=16` (keep `lora_alpha=2*r` each time).
- How does accuracy change?
- How does the trainable parameter count change?
- Is there a point of diminishing returns?

```python
# Try: r=4, lora_alpha=8
# Try: r=16, lora_alpha=32
```

---

### Exercise 2: More target modules

Add `"k_lin"` and `"out_lin"` to `target_modules`. This targets all four attention projections (query, key, value, output). Does it improve accuracy? By how much does it increase the parameter count?

```python
target_modules=["q_lin", "k_lin", "v_lin", "out_lin"]
```

---

### Exercise 3: Scaling factor α

The effective learning rate scaling is `α/r`. Compare:
- `lora_alpha=8` → scale = 1.0 (i.e., no scaling)
- `lora_alpha=16` → scale = 2.0 (default above)
- `lora_alpha=32` → scale = 4.0

Holding `r=8` fixed, how does `lora_alpha` affect convergence speed and final accuracy?

---

### Exercise 4: Merge and unload

PEFT provides `merge_and_unload()` to fuse the LoRA adapter weights back into the base model:

```python
merged_model = lora_model.merge_and_unload()
```

After merging:
1. Verify that `merged_model` produces identical predictions to `lora_model` on the test set.
2. Check `count_params(merged_model, only_trainable=True)` — why is this now equal to the full parameter count?
3. Discuss the tradeoff: when would you merge, and when would you keep the adapter separate?

---

### Summary of Key Numbers from This Notebook

| | Value |
|---|---|
| DistilBERT base params | ~66 million |
| LoRA trainable params (r=8) | <1% of base |
| LoRA test accuracy | ~89% |
| Adapter file size | ~2–3 MB |